# Group 1 — DDSP Preprocessing

Download pretrained DDSP models from GCS, extract f0 (CREPE via torchcrepe) and
A-weighted loudness features at 16 kHz, and chunk audio into overlapping training
examples for fine-tuning.

**Prerequisites:** Install DDSP dependencies (see cell below).

**Outputs:** Preprocessed feature dicts (`audio`, `f0_hz`, `f0_confidence`, `loudness_db`)
and chunked training examples ready for the fine-tuning notebook.

In [1]:
import sys
from pathlib import Path

# Add notebooks/ to path so we can import from shared/
sys.path.insert(0, str(Path("..").resolve()))

import os

import matplotlib.pyplot as plt
import numpy as np

from shared.ddsp import (
    AVAILABLE_MODELS,
    DDSP_SAMPLE_RATE,
    FRAME_RATE,
    _checkpoint_exists,
    chunk_for_training,
    download_model,
    list_base_models,
    preprocess_audio,
)

# --------------- Configuration (edit these) ---------------
MODELS_DIR = "../../models/ddsp_pretrained"

In [3]:
# ---- Download a model ----

MODEL_NAME = "tenor_saxophone"  # Change to: flute, flute2, trumpet, tenor_saxophone

# Show current download status
for m in list_base_models(MODELS_DIR):
    status = "downloaded" if m["downloaded"] else "not downloaded"
    print(f"  {m['name']:20s} {status}")

# Download if needed
ckpt_dir = os.path.join(MODELS_DIR, MODEL_NAME)
if not _checkpoint_exists(ckpt_dir):
    download_model(MODEL_NAME, ckpt_dir)
else:
    print(f"Model '{MODEL_NAME}' already downloaded at {ckpt_dir}")

  violin               downloaded
  flute                not downloaded
  flute2               not downloaded
  trumpet              not downloaded
  tenor_saxophone      not downloaded
  [1/4] ckpt-20000.data-00000-of-00001
  [2/4] ckpt-20000.index
  [3/4] dataset_statistics.pkl
  [4/4] operative_config-0.gin
Model 'tenor_saxophone' downloaded to ../../models/ddsp_pretrained/tenor_saxophone


In [ ]:
# ---- Run preprocessing ----

WAV_PATH = "../../recordings/your_recording.wav"  # <-- edit this

features = preprocess_audio(WAV_PATH)

# Plot f0 contour and loudness
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

time_frames = np.arange(len(features["f0_hz"])) / FRAME_RATE

axes[0].plot(time_frames, features["f0_hz"], linewidth=0.5)
axes[0].set_ylabel("f0 (Hz)")
axes[0].set_title("Fundamental Frequency")

axes[1].plot(time_frames, features["loudness_db"], linewidth=0.5, color="orange")
axes[1].set_ylabel("Loudness (dB)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("A-Weighted Loudness")

plt.tight_layout()
plt.show()

In [ ]:
# ---- Chunk for training ----

examples = chunk_for_training(features)

if examples:
    ex = examples[0]
    print(f"Example shapes:")
    print(f"  audio:      {ex['audio'].shape}")
    print(f"  f0_hz:      {ex['f0_hz'].shape}")
    print(f"  loudness_db: {ex['loudness_db'].shape}")
else:
    print("No examples generated — recording may be too short (need >= 4s).")